# Predicción del Tipo de Cambio USD/GBP con MLP

En este notebook se implementa un Perceptrón Multicapa (MLP) para predecir el precio de cierre del tipo de cambio USD/GBP utilizando datos históricos.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 2. Carga y exploración de datos

In [ ]:
RUTA = 'USD_GBP Historical Data.csv'
df = pd.read_csv(RUTA)
print(f'Dimensiones del dataset: {df.shape}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Preprocesamiento de datos

In [ ]:
# Convertir la columna Date a datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Convertir columnas numéricas (Price, Open, High, Low)
for col in ['Price', 'Open', 'High', 'Low']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Limpiar la columna Change %
df['Change %'] = df['Change %'].str.replace('%', '').astype(float)

# Limpiar la columna Vol. (puede tener K, M, o estar vacía)
def parse_volume(val):
    if pd.isna(val) or val == '' or val == '-':
        return 0.0
    val = str(val).strip()
    if val.endswith('K'):
        return float(val[:-1]) * 1_000
    elif val.endswith('M'):
        return float(val[:-1]) * 1_000_000
    elif val.endswith('B'):
        return float(val[:-1]) * 1_000_000_000
    else:
        try:
            return float(val)
        except:
            return 0.0

df['Vol.'] = df['Vol.'].apply(parse_volume)

# Ordenar por fecha (de más antigua a más reciente)
df = df.sort_values('Date').reset_index(drop=True)

print('Valores nulos por columna:')
print(df.isnull().sum())
print(f'\nRegistros totales: {len(df)}')
df.head()

## 4. Visualización de los datos

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Price'], color='steelblue', linewidth=0.8)
plt.title('Tipo de Cambio USD/GBP - Serie Temporal', fontsize=14)
plt.xlabel('Fecha')
plt.ylabel('Precio (USD/GBP)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Preparación de características (features)

Se utilizan ventanas temporales (time steps) para crear las características de entrada al MLP.

In [ ]:
# Usar el precio de cierre como variable objetivo
data = df['Price'].values.reshape(-1, 1)

# Normalizar los datos entre 0 y 1
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Crear secuencias con ventana temporal
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size, 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

WINDOW_SIZE = 10  # Usar los últimos 10 días para predecir el siguiente
X, y = create_sequences(data_scaled, WINDOW_SIZE)

print(f'Forma de X: {X.shape}')
print(f'Forma de y: {y.shape}')

## 6. División en conjuntos de entrenamiento y prueba

In [ ]:
# Dividir 80% entrenamiento, 20% prueba (sin mezclar por ser serie temporal)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba: {X_test.shape[0]} muestras')

## 7. Entrenamiento del MLP

In [ ]:
# Crear y entrenar el modelo MLP
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate='adaptive',
    learning_rate_init=0.001
)

mlp.fit(X_train, y_train)
print(f'Iteraciones realizadas: {mlp.n_iter_}')
print(f'Pérdida final: {mlp.loss_:.6f}')

## 8. Curva de pérdida durante el entrenamiento

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(mlp.loss_curve_, color='crimson')
plt.title('Curva de Pérdida durante el Entrenamiento', fontsize=14)
plt.xlabel('Iteraciones')
plt.ylabel('Pérdida (MSE)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Predicciones y evaluación

In [ ]:
# Predicciones
y_pred_train = mlp.predict(X_train)
y_pred_test = mlp.predict(X_test)

# Desnormalizar para obtener valores reales
y_train_real = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_train_real = scaler.inverse_transform(y_pred_train.reshape(-1, 1)).flatten()
y_pred_test_real = scaler.inverse_transform(y_pred_test.reshape(-1, 1)).flatten()

# Métricas de evaluación
print('=== Métricas en Entrenamiento ===')
print(f'MSE:  {mean_squared_error(y_train_real, y_pred_train_real):.6f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_train_real, y_pred_train_real)):.6f}')
print(f'MAE:  {mean_absolute_error(y_train_real, y_pred_train_real):.6f}')
print(f'R²:   {r2_score(y_train_real, y_pred_train_real):.6f}')

print('\n=== Métricas en Prueba ===')
print(f'MSE:  {mean_squared_error(y_test_real, y_pred_test_real):.6f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test_real, y_pred_test_real)):.6f}')
print(f'MAE:  {mean_absolute_error(y_test_real, y_pred_test_real):.6f}')
print(f'R²:   {r2_score(y_test_real, y_pred_test_real):.6f}')

## 10. Visualización de resultados

In [ ]:
# Gráfica de predicciones vs valores reales (conjunto de prueba)
plt.figure(figsize=(14, 5))
plt.plot(y_test_real, label='Valor Real', color='steelblue', linewidth=1)
plt.plot(y_pred_test_real, label='Predicción MLP', color='crimson', linewidth=1, linestyle='--')
plt.title('Predicción vs Valor Real - Conjunto de Prueba', fontsize=14)
plt.xlabel('Muestras')
plt.ylabel('Precio USD/GBP')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica completa: entrenamiento + prueba
plt.figure(figsize=(14, 5))
plt.plot(range(len(y_train_real)), y_train_real, label='Entrenamiento (Real)', color='steelblue', linewidth=0.8)
plt.plot(range(len(y_train_real), len(y_train_real) + len(y_test_real)), y_test_real, label='Prueba (Real)', color='green', linewidth=0.8)
plt.plot(range(len(y_train_real), len(y_train_real) + len(y_pred_test_real)), y_pred_test_real, label='Prueba (Predicción)', color='crimson', linewidth=0.8, linestyle='--')
plt.axvline(x=len(y_train_real), color='gray', linestyle=':', label='Inicio de Prueba')
plt.title('Predicción del Tipo de Cambio USD/GBP con MLP', fontsize=14)
plt.xlabel('Muestras')
plt.ylabel('Precio USD/GBP')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica de dispersión: predicción vs real
plt.figure(figsize=(6, 6))
plt.scatter(y_test_real, y_pred_test_real, alpha=0.5, s=10, color='steelblue')
min_val = min(y_test_real.min(), y_pred_test_real.min())
max_val = max(y_test_real.max(), y_pred_test_real.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
plt.title('Dispersión: Valor Real vs Predicción', fontsize=14)
plt.xlabel('Valor Real')
plt.ylabel('Predicción')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Conclusiones

Se implementó un modelo MLP para predecir el tipo de cambio USD/GBP. El modelo utiliza una ventana de 10 días para predecir el precio del día siguiente. Las métricas R² y RMSE permiten evaluar la calidad del modelo.